# Timestamp Variation — `restaurant_offered_timestamp_utc` vs `eater_request_timestamp_local`

`region` holds a single value ("Mexico") so the per-zone mapping is done on `territory` instead,
which has five distinct values across Mexico's two active time zones.

In [ ]:
import pandas as pd
from pathlib import Path

RAW_PATH = Path('../data/raw/BC_A&A_with_ATD.csv')

df = pd.read_csv(
    RAW_PATH,
    usecols=[
        'territory',
        'workflow_uuid',
        'delivery_trip_uuid',
        'restaurant_offered_timestamp_utc',
        'eater_request_timestamp_local',
    ],
    parse_dates=[
        'restaurant_offered_timestamp_utc',
        'eater_request_timestamp_local',
    ],
    na_values=['\\N', 'NULL', 'None', ''],
)

print(f'Rows loaded: {len(df):,}')
print()
print('Territory distribution:')
print(df['territory'].value_counts().to_string())

In [ ]:
df = df.dropna(subset=[
    'restaurant_offered_timestamp_utc',
    'eater_request_timestamp_local',
    'territory',
])

# West median was exactly -60 min with +7, meaning the dataset stores
# all local timestamps as UTC-6 regardless of territory.
# Corrected to +6 across the board.
UTC_OFFSET_HOURS = {
    'Central'           : 6,
    'North'             : 6,
    'West'              : 6,
    'South East'        : 6,
    'Long Tail - Region': 6,
}

df['utc_offset_h'] = df['territory'].map(UTC_OFFSET_HOURS)

df['eater_request_utc'] = (
    df['eater_request_timestamp_local']
    + pd.to_timedelta(df['utc_offset_h'], unit='h')
)

print('Offset applied per territory:')
print(df.groupby('territory')['utc_offset_h'].first().to_string())

In [ ]:
# Both columns now in UTC — compare directly
df['variation_min'] = (
    df['restaurant_offered_timestamp_utc'] - df['eater_request_utc']
).dt.total_seconds() / 60

print('Variation (restaurant_offered_utc − eater_request_utc) in minutes\n')
print(df['variation_min'].describe(percentiles=[.05, .25, .5, .75, .95, .99]).round(4))
print(f'\nMean variation : {df["variation_min"].mean():.4f} min')
print(f'Negative rows  : {(df["variation_min"] < 0).sum():,}  '
      f'({(df["variation_min"] < 0).mean()*100:.2f}%)')

print('\n--- Per territory ---')
print(
    df.groupby('territory')['variation_min']
    .agg(['mean', 'median', 'std', 'count'])
    .round(4)
    .to_string()
)

In [ ]:
top10 = (
    df.assign(abs_variation=df['variation_min'].abs())
    .nlargest(10, 'abs_variation')
    [[
        'territory',
        'workflow_uuid',
        'delivery_trip_uuid',
        'eater_request_utc',
        'restaurant_offered_timestamp_utc',
        'variation_min',
        'abs_variation',
    ]]
    .reset_index(drop=True)
)
top10.index += 1
top10.round(4)

In [ ]:
import matplotlib.pyplot as plt

territories = df['territory'].dropna().unique()
n = len(territories)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), sharey=False)

for ax, terr in zip(axes, sorted(territories)):
    subset = df.loc[df['territory'] == terr, 'variation_min']
    clipped = subset.clip(subset.quantile(0.01), subset.quantile(0.99))
    ax.hist(clipped, bins=60, edgecolor='none', color='steelblue')
    ax.axvline(subset.mean(), color='red', linewidth=1.5,
               label=f'mean={subset.mean():.1f}')
    ax.axvline(0, color='black', linewidth=1, linestyle='--')
    ax.set_title(terr, fontsize=9)
    ax.set_xlabel('variation (min)')
    ax.legend(fontsize=8)

fig.suptitle(
    'restaurant_offered_local − eater_request_local  (p1–p99 clipped)',
    fontweight='bold'
)
plt.tight_layout()
plt.show()